## Meter Maintenance Notebook

This notebook is a **review and maintenance tool** for the running broken meter list. It is not the normal Harvest kWh processing pipeline.

Use it when a meter needs closer inspection or when an existing broken meter record may need to be **updated**, **removed**, or **added**.

<span style="color:royalblue">What this notebook does:</span>
1. Loads raw 15 minute cumulative kWh readings.
2. Keeps meters whose `end_use` is exactly `main` or `submeter`.
3. Excludes the configured PV-building main meters and Student Health.
4. Loads the current broken meter list and supporting review files. 
5. Builds a list of meters to review.
6. Creates a PDF showing raw readings and all relevant issue overlays.
7. Creates a maintenance update log (rebuilt from scratch when `clear_update_log = True`).
8. Applies only approved log actions to a **reviewed copy** of the broken meter list.
9. Creates a second PDF preview of the approved changes.
10. Commits the reviewed copy to the source file only after an explicit final switch is set to `True`.



<span style="color:royalblue">Important safety rule:</span>

Running all cells does **not** overwrite `running_list_broken_meters.csv` while: 
```python
commit_reviewed_changes_to_source = False 
```
<span style="color:royalblue">**Keep setting == `False` until the approved change preview PDF has been reviewed**</span>

### Files used by this notebook

<span style="color:royalblue">Inputs:</span>
- `running_list_broken_meters.csv` = source of truth for broken-meter periods.
- `meter_info.csv` = identifies `main` and `submeter` meters.
- Raw Harvest kWh CSV = provides meter readings for review plots.

<span style="color:royalblue">Main pipeline files: </span>
- `special_meters_corrections_master_sheet.csv` remove windows are applied to the loaded data to rebuild the removed readings in memory, so `removed_special_meter_data.csv` is not read or overwritten here.
- `special_meters_corrections_master_sheet.csv` = combined correction records used for reference.

<span style="color:royalblue">Notebook outputs: </span>
- `broken_meter_update_log.csv` = proposed `update`, `remove`, and `add` actions.
- `running_list_broken_meters_reviewed_copy.csv` = preview of the updated broken-meter list.
- `maintenance_review_selected_meters.pdf` = review plots with issue overlays.
- `maintenance_approved_changes_preview.pdf` = final preview before approved changes are committed.

In [39]:
# make plot legend a table with color


In [40]:
# Future cleanup ideas only; these comments do not affect execution.

# important:
# TODO: have it change the updated data date column when applying changes

### 1. Parameters

In [ ]:
############ CHANGE PARAMETERS AS NEEDED #############

data_input_file = 'aurora_kwh_15min_250601-261001.csv'
schema = 'aurora_v4' #'harvest'

# Set to True to delete the previous update log and rebuild it
# (set False after editing it, otherwise your edits are lost)
clear_update_log = True

# set to True only after broken_meter_update_log.csv has been
# reviewed and the desired rows marked approved = 1 (with clear_update_log = False)
update_log_reviewed = False

# set True to commit changes to running_list_broken_meters.csv 
# after confirming maintenance_approved_changes_preview.pdf and running_list_broken_meters_reviewed_copy.csv
commit_reviewed_changes_to_source = False


########## EXTRA PARAMETERS (optional) ##############

# review every meter from the broken source file that has a status in BROKEN_STATUS_VALUES
review_all_broken_status_meters = True

# manual review list, this is used only when review_all_broken_status_meters = False
meters_to_review = [] # leave empty if using automatic broken meter selection

# optional plot zoom, leave both as None to display the full raw-data range.
zoom_start = None  # e.g. "2025-08-01 00:00:00"
zoom_end = None     # e.g. "2025-10-01 00:00:00"

######################################################

In [ ]:
# Data Directories
input_dir = "../data/inputs/"  # raw interval-reading data files
extracts_dir = "../data/extracts/"  # meter info & review lists
output_dir = "../data/outputs/"
maintenance_dir = "../data/outputs/maintenance/"


# Raw cumulative kWh readings used for visual review
var_file = input_dir + data_input_file

# Main pipeline source/review files
broken_meters_file = extracts_dir + "running_list_broken_meters.csv"
meter_info_file = extracts_dir + "meter_info.csv"

# Main pipeline generated files
meter_corrections_file = output_dir + "special_meters_corrections_master_sheet.csv"
# removed_special_meter_data_file = output_dir + "removed_special_meter_data.csv"

# Maintenance files
broken_meter_update_log_file = maintenance_dir + "broken_meter_update_log.csv"
broken_meter_reviewed_copy_file = maintenance_dir + "running_list_broken_meters_reviewed_copy.csv"
maintenance_plot_file = maintenance_dir + "maintenance_review_selected_meters.pdf" #plot_dir + "maintenance_review_selected_meters.pdf"

###################### Thresholds ####################

# Boundary check settings (step 5), compare each broken row's dates with the readings
# bad reading = missing (NaN/0) or part of a flat run at least boundary_min_flat_run long
boundary_min_gap = "1D"          # min length of a gap/good stretch at a boundary to flag (e.g. "12h", "1D", "3D")
boundary_min_flat_run = "1D"     # identical readings this long count as stuck (bad)
boundary_max_restarts = 5        # meters with more than this many restarts in the data are skipped (same as restarts_thres in notebook 1)

# Data gap settings (step 4), long missing stretches on ANY meter in the data, not just broken-list meters
gap_min_length = "30D"           # flag missing stretches at least this long
gap_merge_within = "1D"          # missing stretches separated by fewer readings than this are merged into one gap
gap_min_missing_fraction = 0.9   # a merged gap must still be at least this fraction missing

######################################################
# Exclude: Meters of buildings equipped with PV and Student Health
meters_with_pv = [
    'bachman_hall_main',
    'campus_ctr_main',
    'dance_bldg_main',
    'gartley_hall_main',
    'warrior_rec_ctr_main'
]
meters_excluded = meters_with_pv + ['student_health_main']  # student_health data is in vitality_v5

# data_source for the update log / broken list, from the schema name
# e.g. 'aurora_v4' -> 'aurora', 'harvest' -> 'harvest'
data_source = schema.split('_')[0]


1. **Import dependencies**
2. **Load and reshape raw data**
3. **Select main meters and submeters**
4. **Build the review list**
5. **Load broken, master correction, and removed reading files**
6. **Generate initial review plots**
7. **Prepare the editable update log** (with boundary checks)
8. **Apply approved actions to a reviewed copy**
9. **Generate approved change preview plots**
10. **Optionally commit the reviewed copy to the source file**
11. **Rerun the main pipeline**

### 2. Imports

In [43]:
import os, sys
import importlib
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath('..'))
import modules.data_clean as dc # import self-defined module
importlib.reload(dc)

<module 'modules.data_clean' from '/Users/cassiehuber/Documents/GitHub/harvest_kwh_prep/modules/data_clean.py'>

### 3. Load and Reshape Meter Data

In [44]:
# Load raw interval data
raw_df = pd.read_csv(var_file, low_memory=False)
raw_df["datetime"] = pd.to_datetime(raw_df["datetime"])
raw_df = raw_df.dropna(subset=["datetime"]).copy()

# Pivot to wide meter dataframe with every meter as one column (reshape data)
pivoted_df = raw_df.pivot(
    index="datetime",
    columns="meter_name",
    values="meter_reading",
).reset_index()

# Fill missing timestamps to ensure continuous time series
full_df = dc.fill_missing_timestamps(pivoted_df, "15min")

######################################################
# Keep only exact main meters and submeters from meter_info.csv
# Remove excluded meters (meters with PV)

data, meter_type_by_meter, selected_meter_info = (
    dc.select_meter_types_from_info(
        full_df,
        meter_info_file,
        allowed_end_uses=("main", "submeter"),
        excluded_meters=meters_excluded,
    )
)

# Replace all 0s with NaN, same as 1.kwh_end_points, so 0 readings
# are treated as missing instead of plotting as drops to zero
data = data.replace(0, np.nan)

######################################################

# Load broken meter source file
broken_source_df = dc.load_broken_meter_workbook(broken_meters_file)

# Build review meter list automatically from broken source statuses
data_cols_norm = {str(col).strip().lower() for col in data.columns}

all_broken_status_meters = sorted(
    broken_source_df["meter_name"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
    .unique()
    .tolist()
)

meters_not_in_data = sorted(
    [meter for meter in all_broken_status_meters if meter not in data_cols_norm]
)

if review_all_broken_status_meters:
    meters_to_review = sorted(
        [meter for meter in all_broken_status_meters if meter in data_cols_norm]
    )

print("all broken status meters from source file:", len(all_broken_status_meters))
print("broken status main/submeters not found in selected data:", len(meters_not_in_data))

if meters_not_in_data:
    print("not in selected main/submeter data:")
    print(meters_not_in_data)

######################################################

# Load generated master corrections file
master_corrections_df = pd.read_csv(meter_corrections_file, low_memory=False)

######################################################

# Rebuild removed raw data from the master sheet + the data loaded above
# (not read from removed_special_meter_data.csv, which only matches the data
# notebook 1 last ran on). output_csv=None so the main pipeline file is not overwritten.
removed_df = dc.export_removed_special_meter_data(
    data,
    meter_corrections_file,
    output_csv=None,
)
for col in ["datetime", "correction_start", "correction_end"]:
    if col in removed_df.columns:
        removed_df[col] = pd.to_datetime(removed_df[col], errors="coerce")

######################################################

# Summary of loaded data
print("\nraw_df shape:", raw_df.shape)
print("selected maintenance data shape:", data.shape)
print("broken_source_df rows:", len(broken_source_df))
print("removed_df rows:", len(removed_df))


Selected 136 meters: 106 main, 30 submeter.
all broken status meters from source file: 36
broken status main/submeters not found in selected data: 3
not in selected main/submeter data:
['campus_center_ac_main', 'law_clinic_main', 'law_school_ac_main']

raw_df shape: (5814984, 3)
selected maintenance data shape: (46796, 136)
broken_source_df rows: 36
removed_df rows: 295411


### 4. Meter Review Plots

- One PDF page per broken meter, then a **Data Gap Candidates** section
- Titles show `[main]` or `[submeter]`
- 0 readings are treated as missing
- Use `zoom_start` / `zoom_end` to zoom in on specific timeframes of the data

<span style="color:royalblue">Plot legend: </span>
| Symbol | Meaning |
|:---:|---|
| <span style="color:#1f77b4">━━</span> | raw meter readings |
| <span style="background-color:rgba(255,0,0,0.2); border:1px solid red; padding:0 14px">&nbsp;</span> | broken interval from the broken list |
| <span style="background:repeating-linear-gradient(45deg, rgba(255,0,255,0.6) 0 2px, transparent 2px 6px); border:1px solid magenta; padding:0 14px">&nbsp;</span> | boundary fix: from the current broken date to the suggested date |
| <span style="color:magenta">╌╌</span> | suggested start/end date, labeled with the check name |
| <span style="background-color:rgba(128,0,128,0.2); border:1px solid purple; padding:0 14px">&nbsp;</span> | data gap candidate |
| <span style="color:black">╌╌</span> | restart or drop (reading goes down) |
| <span style="color:orange">●</span> | removed reading (broken list or approved correction) |
| <span style="color:dimgray">●</span> | removed reading, auto-fixed (stuck run or recovered drop) |
- Boundary checks <span style="color:magenta">(magenta)</span>: flag broken rows whose start or end date doesn't match the readings. Step 5 explains each check.

<span style="color:royalblue">Data gap candidates:</span>
- Meters missing readings for `gap_min_length` or longer (including all-NaN meters) where the gap is **not** on the broken list. 
- A broken meter can show up twice: once with the broken meters, and again here for a gap outside its broken rows.


In [45]:
# Check each broken row's start/end dates against the readings
boundary_checks_df = dc.detect_broken_boundary_issues(
    data=data,
    broken_df=broken_source_df,
    meters=meters_to_review,
    min_boundary_gap=boundary_min_gap,
    min_flat_run=boundary_min_flat_run,
    max_restarts=boundary_max_restarts,
)

print(f"Boundary checks: {len(boundary_checks_df)} flags on "
      f"{boundary_checks_df['meter_name'].nunique()} meters (shown in magenta on the plots)")

# Long gaps of missing readings on every meter in the data, skipping gaps already on the broken list
gap_df = dc.detect_data_gaps(
    data=data,
    broken_df=broken_source_df,
    min_gap=gap_min_length,
    merge_within=gap_merge_within,
    min_missing_fraction=gap_min_missing_fraction,
)

print(f"\nData gaps: {len(gap_df)} gaps of {gap_min_length}+ on {gap_df['meter_name'].nunique()} meters "
      f"({gap_df['in_broken_list'].sum()} on meters already in the broken list, shown in purple after the broken meters)")
display(gap_df[["meter_name", "gap_start", "gap_end", "gap_days", "missing_pct", "gap_type", "shared_meters"]])


Boundary checks skipped 7 meters with more than 5 restarts in the data:
  hper_klum_gym: 25 restarts
  malama_1_2_ehso_main: 20 restarts
  pbrc_main_b: 17 restarts
  les_murakami_stadium_main: 16 restarts
  hawaii_hall_main: 14 restarts
  pbrc_main_a: 9 restarts
  hig_substation_2_main: 8 restarts
Boundary checks: 18 flags on 14 meters (shown in magenta on the plots)

Data gaps: 19 gaps of 30D+ on 19 meters (1 on meters already in the broken list, shown in purple after the broken meters)


,meter_name,gap_start,gap_end,gap_days,missing_pct,gap_type,shared_meters
0,andrews_amp_main,2025-06-01 00:00:00,2025-11-09 23:45:00,162.0,91.7,leading,0
1,biomedical_science_ch_2,2026-06-08 08:00:00,2026-07-21 09:30:00,43.1,99.2,internal,0
2,gilmore_hall_main_a,2026-06-16 12:00:00,2026-07-20 10:00:00,33.9,100.0,internal,5
3,hamilton_lib_ph_iii_ch_1,2025-06-26 09:00:00,2025-10-19 05:00:00,114.8,99.4,internal,0
4,hamilton_lib_ph_iii_ch_2,2025-07-23 07:45:00,2025-10-23 11:15:00,92.2,99.4,internal,0
5,hamilton_lib_ph_iii_ch_3,2025-06-01 00:00:00,2025-11-03 09:00:00,155.4,91.7,leading,0
6,hamilton_lib_ph_iii_main_1,2025-07-23 07:45:00,2025-10-19 05:00:00,87.9,100.0,internal,3
7,hamilton_lib_ph_iii_main_2,2025-07-23 07:45:00,2025-10-19 05:00:00,87.9,100.0,internal,3
8,hamilton_lib_ph_iii_main_3,2025-06-01 00:00:00,2025-11-11 14:45:00,163.6,97.2,leading,0
9,hamilton_lib_ph_iii_mcc_ac1,2025-07-23 07:45:00,2025-10-19 05:00:00,87.9,100.0,internal,3


In [46]:
# Generate one PDF page per selected meter, preceded by a standalone legend page.
maintenance_saved_count = dc.plot_meter_maintenance_review_pdf(
    data=data,
    meters_to_review=meters_to_review,
    broken_df=broken_source_df,
    removed_df=removed_df,
    plot_file=maintenance_plot_file,
    meter_type_map=meter_type_by_meter,
    zoom_start=zoom_start,
    zoom_end=zoom_end,
    ylabel="kWh",
    boundary_df=boundary_checks_df,
    gap_df=gap_df,
    #show_plots=False, # can be set to False if running on server and not wanting to display plots
)

Saved 33 broken meter plots and 19 data gap plots to ../data/outputs/maintenance/maintenance_review_selected_meters.pdf


### 5. Create Maintenance Update Log

`broken_meter_update_log.csv` lists proposed changes to the broken meter list. This step does **not** change the broken meter list.

- `clear_update_log = True`: deletes the old log and builds a new one (like `Clear_Candidates` in notebook 1)
- `clear_update_log = False`: uses the log as you saved it

<span style="color:royalblue">The log has two kinds of rows:</span>
1. **Broken meter rows**: one per row in the broken list, alphabetical. The `source_*` columns show the current row, so don't edit them.
2. **Data gap rows** (at the end): one per data gap candidate, prefilled as `action = add`, `new_status = broken`. `new_description` explains the gap.

Nothing is applied until you set `approved = 1` (step 6).

<span style="color:royalblue">Columns:</span>
| Column | Meaning |
|---|---|
| `meter_name` | Meter the row is about |
| `source_start_date` / `source_end_date` | Current dates of the broken list row |
| `source_status` / `source_description` | Current status and description of the broken list row |
| `data_source` | Set from the schema name (e.g. `aurora_v4` → `aurora`) |
| `boundary_check` | Which boundary checks flagged the row (blank = none, see below) |
| `action` | `update`, `remove`, or `add` |
| `new_start_date` / `new_end_date` | New dates (blank = keep the current one) |
| `new_status` / `new_description` | New status and description (blank = keep the current one) |
| `approved` | `1` = apply this row, `0` = skip it |

<span style="color:royalblue">Boundary checks (`boundary_check` column):</span>
- Checks if a broken row's dates match the readings. When a row is flagged, `action = update` and the suggested date are prefilled.
- No date is suggested when the row goes past the data or two checks disagree. Meters with more than `boundary_max_restarts` restarts are skipped.

| Check | Meaning | Suggests |
|---|---|---|
| `start_too_late` | readings went bad before the broken start | earlier start |
| `start_too_early` | readings were still good after the broken start | later start |
| `end_too_early` | readings stayed bad after the broken end | later end |
| `end_too_late` | readings were good again before the broken end | earlier end |
| `data_resumed` | open-ended row, but readings came back | end date |


In [47]:
# clear_update_log = True: delete the old log and rebuild it from scratch with this run's
# broken meter rows (alphabetical), boundary suggestions (from step 4), and data gap
# `add` rows after all the broken meter rows.
# clear_update_log = False (or update_log_reviewed = True): use the log exactly as saved,
# so your edits and approvals are applied as is. A missing log is always built.
if clear_update_log and update_log_reviewed:
    print("update_log_reviewed = True, so clear_update_log is ignored and the reviewed log is kept.")

update_log_df = dc.prepare_broken_meter_update_log(
    update_log_file=broken_meter_update_log_file,
    broken_source_df=broken_source_df,
    meters_to_review=meters_to_review,
    clear_existing=clear_update_log and not update_log_reviewed,
    data_source=data_source,
    boundary_checks_df=boundary_checks_df,
    gap_df=gap_df,
)

# Display only the currently selected meters when possible
print("Update log rows for the selected meters:")
display(
    update_log_df[
        update_log_df["meter_name"].isin(
            [m.strip().lower() for m in meters_to_review]
        )
    ] if meters_to_review else update_log_df.head(2)
)

Boundary checks: 14 update log rows flagged, 12 prefilled with suggested dates (approved left at 0).
Rebuilt update log from scratch: 33 broken meter rows + 19 data gap rows -> ../data/outputs/maintenance/broken_meter_update_log.csv
Update log rows for the selected meters:


,meter_name,source_start_date,source_end_date,source_status,source_description,data_source,boundary_check,action,new_start_date,new_end_date,new_status,new_description,approved
0,ag_engineering_main,2025-02-19 00:00:00,2025-09-03 00:00:00,broken,3/16/26 Vernon says it needs to be reprogrammed,aurora,,,,,,,0
1,ag_engineering_mcc,2025-02-19 00:00:00,2025-09-03 00:00:00,broken,repaired,aurora,,,,,,,0
2,burns_hall_main,2026-04-26 00:00:00,,broken,,aurora,,,,,,,0
3,ching_complex_main,2025-06-14 00:00:00,,broken,,aurora,,,,,,,0
4,gilmore_hall_main_b,2025-08-19 00:00:00,,broken,,aurora,start_too_late,update,2025-08-12 16:30:00,,,,0
5,gilmore_hall_mcc,2025-08-19 00:00:00,,broken,,aurora,start_too_late,update,2025-08-12 16:30:00,,,,0
6,hale_kuahine_main,2026-06-21 00:00:00,,broken,,aurora,,,,,,,0
7,hale_noelani_all_towers_main,2026-03-13 00:00:00,,broken,,aurora,"start_too_early, data_resumed",update,2026-03-14 20:30:00,2026-07-19 23:45:00,,,0
8,hale_noelani_tower_a_b,2024-07-06 00:00:00,,broken,kWh got weird after this date; then flatlined ...,aurora,start_too_early,,,,,,0
9,hale_noelani_tower_b,2021-07-01 00:00:00,,broken,negative kW numbers,aurora,start_too_early,,,,,,0


### 6. Edit/Review Update Log and Review PDF
(`broken_meter_update_log.csv` and `maintenance_review_selected_meters.pdf`)

Open the CSV. For each row you want to apply, set `action` and `approved = 1`. Rows with `approved = 0` are skipped. Boundary check rows (`update`) and data gap rows (`add`) come prefilled, so check them against the plots, then approve, edit, or leave them at 0.

| Action | What it does | What to fill in |
|---|---|---|
| `update` | changes an existing broken list row | only the `new_*` columns you want to change (blank = keep the current value) |
| `remove` | deletes an existing broken list row | nothing else |
| `add` | adds a new broken list row | `meter_name`, `new_status`, dates (blank start = no start, blank end = still broken), `new_description` (optional) |

- **Don't edit the `source_*` columns**. They must match **exactly one** row in `running_list_broken_meters.csv` or the apply step errors. Leave them blank for `add`.
- `new_status`: `broken`, `repaired`, `under_renovation`, or `no_meter`.
- Dates: `YYYY-MM-DD`, `YYYY-MM-DD HH:MM:SS`, `M/D/YYYY`, or `M/D/YYYY HH:MM:SS`.
- `data_source` is prefilled from the schema. `updated_data` is set automatically.
- Example: a meter was fixed on Aug 12 → `update`, `new_end_date = 2025-08-12`, `new_status = repaired`, `approved = 1`.

When done, set `clear_update_log = False` and `update_log_reviewed = True`, then run the notebook again.

In [48]:
if not update_log_reviewed:
    raise SystemExit(
        "Execution stopped because update_log_reviewed = False. "
        "Review broken_meter_update_log.csv, mark the desired rows approved = 1, "
        "then set `update_log_reviewed = True` and re-run the notebook."
    )

SystemExit: Execution stopped because update_log_reviewed = False. Review broken_meter_update_log.csv, mark the desired rows approved = 1, then set `update_log_reviewed = True` and re-run the notebook.

/Users/cassiehuber/miniconda3/envs/harvest/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


### 7. Apply Approved Update Log Changes to Review **Copy**
- only rows with approved == 1 are applied
- it always writes **`running_list_broken_meters_reviewed_copy.csv`** for final review


Notes:
`update`
- finds exactly one matching source row in running_list_broken_meters.csv
- changes only the new_* fields you filled in
- blank new_* fields keep the original source values
- `remove`
finds exactly one matching source row
removes it from the reviewed output
- `add`
creates a new row using meter_name plus the new_* fields
requires new_status
prevents adding an exact duplicate row



In [ ]:
# Apply approved actions and save the safe reviewed copy
reviewed_broken_df, applied_change_log_df = (
    dc.apply_broken_meter_update_actions(
        broken_source_df=broken_source_df,
        update_log_df=update_log_df,
        reviewed_copy_file=broken_meter_reviewed_copy_file,
    )
)

if applied_change_log_df.empty:
    print("No approved update log rows to apply.")
else:
    print("Approved update log rows applied to reviewed copy of broken meters source file:")
    display(applied_change_log_df)

    print("First 20 rows of the reviewed copy of broken meters source file:")
    display(reviewed_broken_df.head(20))

Reviewed broken meter copy saved to ../data/outputs/maintenance/running_list_broken_meters_reviewed_copy.csv
No approved update log rows to apply.


### 8. Preview approved changes visually
- This PDF shows every meter affected by an approved update log action
- Plots of all meters with approved actions and new_*, even if write_changes_to_source_file = False

In [ ]:
# Post maintenance visual check
# Shows raw readings with the newly reviewed broken meter intervals.
# Candidate and removed data overlays are intentionally omitted because
# they have not yet been regenerated by the main pipeline.

proposed_update_plot_file = (maintenance_dir + "maintenance_approved_changes_preview.pdf")

saved_count = 0

if applied_change_log_df.empty:
    print(
        "No approved maintenance changes were applied, "
        "so no post update plots were created."
    )
else:
    # A meter can have multiple approved actions, unique() ensures it receives
    # only one preview page
    changed_meters = sorted(
        applied_change_log_df["meter_name"]
        .dropna()
        .astype(str)
        .str.strip()
        .str.lower()
        .unique()
        .tolist()
    )

    # The preview uses the proposed reviewed broken meter dataframe, not the
    # original source dataframe
    saved_count = dc.plot_approved_maintenance_preview_pdf(
        data=data,
        changed_meters=changed_meters,
        reviewed_broken_df=reviewed_broken_df,
        plot_file=proposed_update_plot_file,
        meter_type_map=meter_type_by_meter,
        zoom_start=zoom_start,
        zoom_end=zoom_end,
        ylabel="kWh",
        show_plots=True # can be set to False if running on server and not wanting to display plots,
    )


No approved maintenance changes were applied, so no post update plots were created.


### 9. Final Commit Reviewed Changes to **Source File**
(`running_list_broken_meters_reviewed.csv`)

COMMIT PROCEDURE:
1. Review broken_meter_update_log.csv
2. Run the approved-action cells
3. Review maintenance_approved_changes_preview.pdf
4. Confirm running_list_broken_meters_reviewed_copy.csv
5. Change the switch in parameters to True
6. running_list_broken_meters.csv updated!

In [ ]:
if not commit_reviewed_changes_to_source:
    print(
        "Preview only. Source broken meter file was not changed.\n"
        "Review the proposed change graph, then set "
        "commit_reviewed_changes_to_source = True and rerun this cell."
    )

elif applied_change_log_df.empty:
    print(
        "No approved update log actions were found. "
        "Source file was not changed."
    )

else:
    # This is the only source file overwrite
    reviewed_broken_df.to_csv(
        broken_meters_file,
        index=False,
        date_format="%Y-%m-%d %H:%M:%S",
    )

    print(
        "Approved maintenance changes committed to source file:\n"
        f"{broken_meters_file}"
    )
    print(
        "Now rerun 1.kwh_end_points so corrections, removed data, "
        "candidates, and final plots regenerate."
    )

if saved_count:
    print(
        f"Review the {saved_count} plots in {proposed_update_plot_file}"
        "before committing reviewed_broken_df to the source broken meter file."
    )

Preview only. Source broken meter file was not changed.
Review the proposed change graph, then set commit_reviewed_changes_to_source = True and rerun this cell.


### 10. Required follow up after committing changes

After `running_list_broken_meters.csv` has been updated, rerun the main `1.kwh_end_points` notebook

- the master correction sheet
- removed special meter data
- unresolved candidate records
- review special meter plots
- corrected all-meter plots
- downstream cleaned and annual meter outputs

Until the main pipeline is rerun, those generated files may still describe the previous broken meter configuration